In [14]:
! pip install langchain_community tiktoken langchainhub chromadb langchain

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the 'C:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


In [6]:
import os
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = "your-api-key"

In [14]:
os.environ['GOOGLE_API_KEY'] = "your-api-key"

In [15]:
!pip install -U langchain-google-genai langchain-text-splitters langchain-community chromadb beautifulsoup4

Defaulting to user installation because normal site-packages is not writeable


You should consider upgrading via the 'C:\Program Files\Python310\python.exe -m pip install --upgrade pip' command.


In [16]:
import bs4
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

In [17]:
#indexing

In [63]:
loader=WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    )
)
docs=loader.load()

In [66]:
# split
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splits=text_splitter.split_documents(docs)
vectorstore = Chroma.from_documents(documents=splits, 
                                   embedding=GoogleGenerativeAIEmbeddings(
   model="models/gemini-embedding-001")
)

retriever = vectorstore.as_retriever()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [67]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using the context below.

Context:
{context}

Question:
{question}

Answer:
""")

In [68]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [69]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [72]:
question = "What is Baahubali?"

docs = retriever.invoke(question)

context = format_docs(docs)

final_prompt = prompt.invoke({
    "context": context,
    "question": question
})

response = llm.invoke(final_prompt)

answer = response.content

print(answer)
print(context)

Baahubali is a film, specifically "Baahubali 2: The Conclusion," which was released in 2017.
Lage Raho Munna Bhai (2006)
Chak De! India (2007)
Oye Lucky! Lucky Oye! (2008)
3 Idiots (2009)
Dabangg (2010)
Azhagarsamiyin Kuthirai (2011)
Vicky Donor and Ustad Hotel (2012)
Bhaag Milkha Bhaag (2013)
Mary Kom (2014)
Bajrangi Bhaijaan (2015)
Sathamanam Bhavati (2016)
Baahubali 2: The Conclusion (2017)
Badhaai Ho (2018)
Maharshi (2019)
Tanhaji (2020)
2021–present
RRR (2021)
Kantara (2022)
Rocky Aur Rani Kii Prem Kahaani (2023)

Lage Raho Munna Bhai (2006)
Chak De! India (2007)
Oye Lucky! Lucky Oye! (2008)
3 Idiots (2009)
Dabangg (2010)
Azhagarsamiyin Kuthirai (2011)
Vicky Donor and Ustad Hotel (2012)
Bhaag Milkha Bhaag (2013)
Mary Kom (2014)
Bajrangi Bhaijaan (2015)
Sathamanam Bhavati (2016)
Baahubali 2: The Conclusion (2017)
Badhaai Ho (2018)
Maharshi (2019)
Tanhaji (2020)
2021–present
RRR (2021)
Kantara (2022)
Rocky Aur Rani Kii Prem Kahaani (2023)

2001 Indian film by Ashutosh Gowarikar
This